In [11]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint, Bounds

## Values
n = 20
d = np.full(n, 225.0)   # demand per household (L/day)
b = np.full(n, 100.0)   # minimum basic water per household (L/day)
S = 3500.0              # total available source water (L/day)

# delivery-loss factor range (terrain/pipe losses)
# Example: a_i in [1.00, 1.25] means 0% to 25% source overhead.
a = np.linspace(1.00, 1.25, n)

# vulnerability priority range (equity weights)
# Higher weight --> shortages for that household are penalized more.
w = np.linspace(1.00, 1.80, n)

#Checks
if n <= 0:
    raise ValueError("n must be positive.")

if np.any(d < b):
    raise ValueError("Demand must be >= minimum basic water for each household.")

if S <= 0:
    raise ValueError("Total supply S must be positive.")

if np.any(a < 1.0):
    raise ValueError("Each delivery factor a_i should be >= 1.0.")

if np.any(w <= 0.0):
    raise ValueError("Each vulnerability weight w_i must be positive.")

#source interval with lower bound b and upper bound d
S_min = np.sum(a * b)
S_max = np.sum(a * d)
print(f"Feasible source interval: [{S_min:.1f}, {S_max:.1f}] L/day")
print(f"Chosen total supply S: {S:.1f} L/day")
print(f"Delivery factor range a: [{np.min(a):.2f}, {np.max(a):.2f}]")
print(f"Vulnerability range w: [{np.min(w):.2f}, {np.max(w):.2f}]")

if S < S_min:
    raise ValueError("Infeasible: supply is below total minimum requirement.")

if S > S_max:
    print("Note: supply is above total demand, some water may remain unused in this model.")

# Objective function 
def objective(x):
    return np.sum(w * (d - x) ** 2)

def objective_grad(x):
    return 2.0 * w * (x - d)

# Constraint and bounds
A = a.reshape(1, -1)
eq_constr = LinearConstraint(A, lb=S, ub=S)

bounds = Bounds(lb=b, ub=d)

# Initial guess 
x0 = np.full(n, S / n)
x0 = np.clip(x0, b, d)


res = minimize(
    fun=objective,
    x0=x0,
    jac=objective_grad,
    method='trust-constr',
    constraints=[eq_constr],
    bounds=bounds,
    options={'verbose': 0}
 )

x_opt = res.x


print(f"Total delivered: {np.sum(x_opt):.2f} L/day")
print(f"Average per household: {np.mean(x_opt):.2f} L/day")
print(f"Households below 100 L/day: {np.sum(x_opt < 100.0)}")

print("\nHousehold allocations (L/day):")
for i in range(min(20, n)):
    print(f"Household {i+1:2d}: {x_opt[i]:.2f}")

Feasible source interval: [2250.0, 5062.5] L/day
Chosen total supply S: 3500.0 L/day
Delivery factor range a: [1.00, 1.25]
Vulnerability range w: [1.00, 1.80]
Total delivered: 3100.73 L/day
Average per household: 155.04 L/day
Households below 100 L/day: 0

Household allocations (L/day):
Household  1: 139.59
Household  2: 141.97
Household  3: 144.16
Household  4: 146.18
Household  5: 148.06
Household  6: 149.81
Household  7: 151.44
Household  8: 152.96
Household  9: 154.39
Household 10: 155.73
Household 11: 156.99
Household 12: 158.18
Household 13: 159.30
Household 14: 160.37
Household 15: 161.37
Household 16: 162.32
Household 17: 163.23
Household 18: 164.09
Household 19: 164.91
Household 20: 165.69


In [ ]:
# ===============================
# SIMULATED ANNEALING 
# ===============================

T_0       = 500.0
k_max     = 50_000
step_size = 20.0

def sa_make_x0():
    x = b.copy()
    remaining = S - np.dot(a, x)
    capacity  = np.dot(a, d - b)
    if remaining > 0 and capacity > 0:
        fill = np.minimum(d - x, remaining * a * (d - b) / capacity)
        x   += fill / a
    return np.clip(x, b, d)

def sa_perturb(x):
    
    #Transfer water between two random households i and j.
    #The delivery factors a_i ensure the source constraint stays satisfied:
    #a_i * (-delta) + a_j * (delta * a_i/a_j) = 0  =>  sum(a*x) unchanged
    
    x_new = x.copy()
    i, j  = np.random.choice(n, size=2, replace=False)

    max_take     = x_new[i] - b[i]
    max_give     = d[j] - x_new[j]
    max_give_src = max_give * a[j] / a[i]

    # FIX 1: limit berechnen vor uniform(), damit delta die Bounds nie verletzt
    limit = min(max_take, max_give_src, step_size)
    if limit < 1e-9:
        return x_new

    delta = np.random.uniform(0, limit)

    x_new[i] -= delta
    x_new[j] += delta * a[i] / a[j]
    return x_new   # FIX 2: kein np.clip — würde sum(a*x)=S brechen

# ── Simulated Annealing — Steps 1–5 from CO3 Chapter 7 ───────────────────────
np.random.seed(42)

x_cur  = sa_make_x0()
f_cur  = objective(x_cur)
x_best = x_cur.copy()
f_best = f_cur

for k in range(k_max):

    # Step 2: choose new candidate (x_test fulfills all constraints)
    x_test = sa_perturb(x_cur)
    f_test = objective(x_test)

    # Step 3: accept if better
    if f_test <= f_cur:
        x_cur, f_cur = x_test, f_test

    # Step 4: accept with probability if worse (Metropolis criterion)
    else:
        T_k = T_0 * (1 - (k + 1) / k_max)      # Step 5: linear cooling schedule
        s   = np.random.rand()                   #         T_{k+1} = T_0*(1-(k+1)/k_max)
        if T_k > 1e-10 and np.exp(-(f_test - f_cur) / T_k) >= s:
            x_cur, f_cur = x_test, f_test

    # Track global best
    if f_cur < f_best:
        f_best = f_cur
        x_best = x_cur.copy()

x_sa = x_best

# ── SA Results & Method Comparison ───────────────────────────────────────────
print(f"\n{'':30s} {'Lagrange-Newton':>15}  {'Simul. Annealing':>16}")
print(f"{'Objective f(x)':30s} {objective(x_opt):>15.4f}  {f_best:>16.4f}")
print(f"{'Total delivered (L/day)':30s} {np.sum(x_opt):>15.2f}  {np.sum(x_sa):>16.2f}")
print(f"{'Constraint |sum(a*x) - S|':30s} {abs(np.dot(a,x_opt)-S):>15.6f}  {abs(np.dot(a,x_sa)-S):>16.6f}")
print(f"{'Fairness std dev (L/day)':30s} {np.std(x_opt):>15.4f}  {np.std(x_sa):>16.4f}")

## Simulated Annealing

Simulated Annealing mimics the physical process of slowly cooling a metal: at high temperature,
atoms move freely and explore many states; as the material cools, it settles into a low-energy
configuration. Applied to our water allocation problem, this means the algorithm starts by
accepting both better and worse distributions (exploration), and gradually becomes more selective
until it converges on a fair allocation (exploitation).

### Key idea: feasibility-preserving moves
The supply constraint `Σ aᵢ xᵢ = 3500 L/day` must hold at every step. Rather than using a
penalty term, each move transfers water directly from one household to another — adjusted by the
pipe-loss factors `aᵢ ∈ [1.00, 1.25]` — so the constraint is never violated in the first place.

### Algorithm
At each of the 50 000 iterations, a new allocation is proposed by randomly shifting up to
20 L/day between two households. If the weighted sum of squared shortfalls `Σ wᵢ(dᵢ − xᵢ)²`
improves, the new allocation is accepted. If it is worse, it is still accepted with a probability
that decreases as the temperature drops — preventing the search from getting stuck early.
The best allocation found across all iterations is returned as the final solution.

### Cooling schedule
The temperature follows a linear decay from 500 to 0 over the run:
`Tₖ = 500 · (1 − k / 50 000)`, as defined in CO3 Chapter 7.
At high temperature, vulnerable households (high `wᵢ`) and less vulnerable ones are treated
almost equally. As temperature falls, the equity weights increasingly shape which reallocations
are accepted — naturally steering the solution towards protecting the most vulnerable.